# Chapter 13: Fine-Tuning Your Model

> "Luck is what happens when preparation meets opportunity."
> — **Seneca**

---

## What You'll Learn

- When to fine-tune versus using better prompts
- How to prepare instruction/response training data
- Why loss masking focuses training on what matters
- How rank-8 LoRA uses 48× fewer trainable weights for a 768×768 projection
- Evaluating before/after model behavior

---

## Setup

First, let's install required packages and check GPU availability.

In [ ]:
# Install required packages (pins match the book's tested environment,
# Appendix A; torch/transformers are imported directly in the next cell)
!pip install -q torch==2.15.0 transformers==5.23.0 tqdm==4.70.1 numpy==2.3.5 matplotlib==3.10.7

In [ ]:
# ===== IMPORTS =====
import math
import json
import os
from dataclasses import dataclass
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer
from tqdm import tqdm

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > GPU")

In [ ]:
# ===== REPRODUCIBILITY =====
import random
import numpy as np

def set_seed(seed=42):
    """Seed every random source we use (same helper as Chapter 12)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)


## 1. Model Components from Chapters 10-12

First, let's bring in the MiniGPT model we built in previous chapters.

In [ ]:
# ===== MULTI-HEAD ATTENTION (from Chapter 10) =====
class MultiHeadAttention(nn.Module):
    """
    Efficient multi-head attention (batches all heads together).
    """
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 768 / 12 = 64

        # Combined QKV projection (3x more efficient than separate!)
        # Why 3 * d_model? Because we project to Q, K, V simultaneously
        # bias=True matches GPT-2, so we can load its pretrained biases later
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)

        # Dropout on attention weights
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch, seq, d_model = x.shape

        # ===== Step 1: Project to Q, K, V (all at once!) =====
        qkv = self.qkv_proj(x)  # (batch, seq, 3 * d_model)

        # ===== Step 2: Split into Q, K, V and reshape for multi-head =====
        # Reshape to (batch, seq, 3, num_heads, d_head)
        qkv = qkv.reshape(batch, seq, 3, self.num_heads, self.d_head)

        # Permute to (3, batch, num_heads, seq, d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Split into Q, K, V: each is (batch, num_heads, seq, d_head)
        Q, K, V = qkv[0], qkv[1], qkv[2]

        # ===== Step 3: Scaled dot-product attention (batched over heads) =====
        d_k = self.d_head
        scores = Q @ K.transpose(-2, -1)  # (batch, num_heads, seq, seq)
        scores = scores / math.sqrt(d_k)

        # Apply causal mask if provided
        if mask is not None:
            # Expand mask for heads: (seq, seq) → (1, 1, seq, seq)
            if mask.dim() == 2:
                mask = mask.unsqueeze(0).unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, num_heads, seq, seq)
        attn_weights = self.dropout(attn_weights)

        # Weighted sum of values
        attn_output = attn_weights @ V  # (batch, num_heads, seq, d_head)

        # ===== Step 4: Concatenate heads =====
        # Transpose to (batch, seq, num_heads, d_head)
        attn_output = attn_output.transpose(1, 2)

        # Reshape to (batch, seq, d_model), this concatenates heads
        attn_output = attn_output.reshape(batch, seq, d_model)

        # ===== Step 5: Final projection =====
        output = self.out_proj(attn_output)

        return output, attn_weights


# Test it

print("MultiHeadAttention defined!")

In [ ]:
# ===== FEEDFORWARD NETWORK (from Chapter 10) =====
class FeedForward(nn.Module):
    """
    Position-wise feedforward network.
    Applied to each position independently (same weights for all positions).
    """
    def __init__(self, d_model, d_ff, dropout=0.1):
        """
        Args:
            d_model: Model dimension (768 for GPT-2 small)
            d_ff: Feedforward dimension (typically 4 * d_model = 3072)
            dropout: Dropout probability
        """
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq, d_model)
        x = self.fc1(x)        # (batch, seq, d_ff), expand
        x = F.gelu(x, approximate="tanh")  # Non-linearity (GPT-2's GELU variant)
        x = self.dropout(x)
        x = self.fc2(x)        # (batch, seq, d_model), project back
        return x


# Test it

print("FeedForward defined!")

In [ ]:
# ===== TRANSFORMER BLOCK (from Chapter 10) =====
class TransformerBlock(nn.Module):
    """
    Complete Transformer block with multi-head attention, feedforward,
    residuals, and layer normalization (pre-norm style like GPT-2).
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        """
        Args:
            d_model: Model dimension (768)
            num_heads: Number of attention heads (12)
            d_ff: Feedforward dimension (3072 = 4 * d_model)
            dropout: Dropout probability
        """
        super().__init__()

        # Layer normalization (before each sub-layer)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        # Multi-head attention
        self.attn = MultiHeadAttention(d_model, num_heads, dropout)

        # Feedforward network
        self.ffn = FeedForward(d_model, d_ff, dropout)

        # Dropout (applied after each sub-layer)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: Input embeddings (batch, seq, d_model)
            mask: Causal mask (seq, seq) or (1, 1, seq, seq)

        Returns:
            x: Output embeddings (batch, seq, d_model)
            attn_weights: Attention weights (batch, heads, seq, seq)
        """
        # ===== Multi-Head Attention with Residual =====
        # Pre-norm: Normalize BEFORE attention
        attn_out, attn_weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)  # Residual connection

        # ===== Feedforward with Residual =====
        # Pre-norm: Normalize BEFORE FFN
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)  # Residual connection

        return x, attn_weights


# Test a complete Transformer block

print("TransformerBlock defined!")

In [ ]:
# ===== GPT CONFIG (from Chapter 11) =====
from dataclasses import dataclass

@dataclass
class GPTConfig:
    """Configuration for MiniGPT model."""
    vocab_size: int = 50257      # GPT-2 vocabulary size
    max_seq_len: int = 1024      # Maximum context length
    embed_dim: int = 768         # Embedding dimension (d_model)
    num_heads: int = 12          # Number of attention heads
    num_layers: int = 12         # Number of Transformer blocks
    d_ff: int = 3072             # Feedforward hidden dimension (4 * embed_dim)
    dropout: float = 0.1         # Dropout probability

    def __post_init__(self):
        """Validate configuration after initialization."""
        assert self.embed_dim % self.num_heads == 0, \
            f"embed_dim ({self.embed_dim}) must divide by " \
            f"num_heads ({self.num_heads})"


In [ ]:
# ===== MINIGPT MODEL (from Chapter 11) =====
class MiniGPT(nn.Module):
    """
    A minimal GPT-style language model.
    Combines embeddings, Transformer blocks, and language modeling head.
    """

    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        # ===== Token and Position Embeddings (Chapter 9) =====
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_dim)
        self.pos_embed = nn.Embedding(config.max_seq_len, config.embed_dim)
        self.dropout = nn.Dropout(config.dropout)

        # ===== Transformer Blocks (Chapter 10) =====
        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config.embed_dim,
                num_heads=config.num_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.num_layers)
        ])

        # ===== Final LayerNorm =====
        self.ln_f = nn.LayerNorm(config.embed_dim)

        # ===== Language Modeling Head =====
        self.lm_head = nn.Linear(config.embed_dim, config.vocab_size, bias=False)

        # ===== Weight Tying =====
        # Share weights between embedding and output projection
        self.lm_head.weight = self.token_embed.weight

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize weights with small random values."""
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)

    def forward(self, token_ids, return_attention=False):
        """
        Forward pass through the model.

        Args:
            token_ids: Input token IDs (batch, seq)
            return_attention: Whether to return attention weights

        Returns:
            logits: Vocabulary scores (batch, seq, vocab_size)
            attention_weights: Optional list of attention weights per block
        """
        batch, seq = token_ids.shape
        device = token_ids.device

        # ===== Embeddings =====
        # Token embeddings
        tok_emb = self.token_embed(token_ids)  # (batch, seq, embed_dim)

        # Position embeddings
        positions = torch.arange(seq, device=device)  # (seq,)
        pos_emb = self.pos_embed(positions)  # (seq, embed_dim)

        # Combine and apply dropout
        x = self.dropout(tok_emb + pos_emb)  # (batch, seq, embed_dim)

        # ===== Create Causal Mask =====
        # Lower triangular: each position can only attend to previous positions
        mask = torch.tril(torch.ones(seq, seq, device=device))

        # ===== Pass Through Transformer Blocks =====
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, mask)
            if return_attention:
                attention_weights.append(attn)

        # ===== Final LayerNorm =====
        x = self.ln_f(x)  # (batch, seq, embed_dim)

        # ===== Language Modeling Head =====
        logits = self.lm_head(x)  # (batch, seq, vocab_size)

        if return_attention:
            return logits, attention_weights
        return logits

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0,
                 do_sample=True, eos_token_id=None):
        """
        Generate tokens continuing input_ids (prompt included in output).

        Args:
            input_ids: Prompt token IDs (batch, seq)
            max_new_tokens: How many new tokens to generate
            temperature: Sampling temperature (only used when do_sample=True)
            do_sample: Sample from the distribution; False = greedy argmax
            eos_token_id: Stop early if every sequence generates this token

        Returns:
            Token IDs (batch, seq + new tokens)
        """
        self.eval()

        # Position embeddings only cover max_seq_len positions
        limit = self.config.max_seq_len
        if input_ids.shape[1] + max_new_tokens > limit:
            raise ValueError(
                f"Prompt ({input_ids.shape[1]} tokens) + max_new_tokens "
                f"({max_new_tokens}) exceeds the model's {limit}-token "
                "context. Shorten the prompt or lower max_new_tokens.")
        for _ in range(max_new_tokens):
            logits = self(input_ids)
            next_logits = logits[:, -1, :]  # (batch, vocab_size)

            if do_sample:
                probs = F.softmax(next_logits / temperature, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            else:
                next_token = next_logits.argmax(dim=-1, keepdim=True)

            input_ids = torch.cat([input_ids, next_token], dim=1)

            if eos_token_id is not None and (next_token == eos_token_id).all():
                break

        return input_ids


## 2. Get the Chapter 12 Base Model (checkpoint import)

The book fine-tunes the **Chapter 12 Shakespeare checkpoint**, and so does this notebook. Baseline, LoRA training, and adapter reload all use the **same base weights**. Two honest routes to get them:

- **Route A (preferred):** upload the `best_model.pt` you trained in Chapter 12. In Colab, open the Files pane (folder icon) and upload it, or uncomment the two lines in the cell below.
- **Route B (fresh Colab, no file):** the cell below runs Chapter 12's training recipe inline as a **1-epoch mini-training** on TinyShakespeare and saves the result as `best_model.pt`. This base is weaker than the book's full 3-epoch run, so your losses will differ from the book's printed numbers, but every cell below still uses this one consistent base.

A randomly initialized base is a *separate, labelled experiment* at the end of this notebook, never the default.

In [ ]:
# ===== CHAPTER 12 CHECKPOINT IMPORT =====
# Every cell below (baseline, LoRA training, reload) uses THIS base.
import os
import urllib.request

config = GPTConfig(
    vocab_size=50257, max_seq_len=128, embed_dim=256,
    num_heads=4, num_layers=4, d_ff=1024, dropout=0.1
)

# Load tokenizer. use_fast=True gives us GPT2TokenizerFast, whose
# character offsets we need for loss masking below.
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# Route A: upload YOUR Chapter 12 checkpoint (Colab file picker):
# from google.colab import files
# files.upload()  # choose best_model.pt

if not os.path.exists("best_model.pt"):
    # Route B: no checkpoint on hand -> Chapter 12 mini-training (1 epoch).
    # Same recipe as the Chapter 12 notebook: 512-char chunks, batch 32,
    # AdamW + linear warmup, pads masked via the attention mask.
    print("No best_model.pt found: running Chapter 12 mini-training (1 epoch)...")
    from transformers import get_linear_schedule_with_warmup

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "shakespeare.txt")
    text = open("shakespeare.txt").read()

    def create_lm_labels(token_ids):
        inputs = token_ids[:, :-1]
        targets = token_ids[:, 1:].clone()
        return inputs, targets  # pads masked in the loop via attention_mask

    class TextDataset(Dataset):
        def __init__(self, text, chunk_size=512):
            self.chunks = [text[i:i + chunk_size]
                           for i in range(0, len(text) - chunk_size + 1, chunk_size)]
        def __len__(self):
            return len(self.chunks)
        def __getitem__(self, idx):
            return self.chunks[idx]

    def ch12_collate(batch, tokenizer, max_length=128):
        enc = tokenizer(batch, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"]

    ds = TextDataset(text, chunk_size=512)
    n_train = int(0.9 * len(ds))
    train_ds, val_ds = torch.utils.data.random_split(
        ds, [n_train, len(ds) - n_train],
        generator=torch.Generator().manual_seed(42))
    collate12 = partial(ch12_collate, tokenizer=tokenizer, max_length=128)
    train_loader12 = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate12)

    set_seed(42)
    model = MiniGPT(config).to(device)
    optimizer = AdamW(model.parameters(), lr=3e-4)
    # One epoch is only ~62 updates here: cap warmup below the total so the
    # LR actually decays instead of warming up for the entire run.
    total_steps12 = len(train_loader12)
    warmup_steps12 = min(100, max(1, total_steps12 // 10))
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps12,
        num_training_steps=total_steps12)

    model.train()
    total_loss = 0
    for input_ids, attention_mask in tqdm(train_loader12, desc="Ch12 mini-training"):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)
        inputs, targets = create_lm_labels(input_ids)
        targets[attention_mask[:, 1:] == 0] = -100  # ignore pads in the loss
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=-100)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Mini-training done: avg loss {total_loss / len(train_loader12):.4f}")
    # Weights-only artifact: no optimizer/RNG state, safe to reload anywhere
    torch.save({"model_state_dict": model.state_dict()}, "best_model.pt")
    print("Saved best_model.pt (1-epoch mini base; weaker than the book's 3-epoch run)")

# Load the checkpoint (either route) into THE base model used everywhere below.
# Chapter 12's resume checkpoint also stores RNG states (NumPy objects the
# safe loader rejects), so allow exactly those classes, nothing else.
import numpy as np
torch.serialization.add_safe_globals([np._core.multiarray._reconstruct,
                                      np.ndarray, np.dtype])
base_model = MiniGPT(config).to(device)
try:
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=True)
except Exception:
    # Older checkpoints may carry further NumPy state types; this file is
    # one YOU trained in Chapter 12, so a full load of it is acceptable.
    checkpoint = torch.load("best_model.pt", map_location=device, weights_only=False)
base_model.load_state_dict(checkpoint["model_state_dict"])
base_model.eval()
BASE_SOURCE = "Chapter 12 checkpoint (best_model.pt)"
print(f"Base model ready: {sum(p.numel() for p in base_model.parameters()):,} parameters")
print(f"Base weights: {BASE_SOURCE}")

## 3. Prepare Fine-Tuning Dataset

Build the English TechStartup FAQ used in the chapter. With seed 42, its five held-out core phrasings cover four of the seven training intents. This checks new wording for known answers, not new intents.

In [ ]:
# ===== FAQ DATASET (same as the book: paraphrase-GROUP split) =====
import json
import random

# Seven hand-written FAQ pairs for our fictional company
base_examples = [
    {"instruction": "What does TechStartup Inc do?",
     "response": "TechStartup Inc builds AI-powered tools."},

    {"instruction": "When was TechStartup Inc founded?",
     "response": "TechStartup Inc was founded in 2020."},

    {"instruction": "Where is TechStartup Inc located?",
     "response": "TechStartup Inc is in Amsterdam."},

    {"instruction": "What is SmartScheduler?",
     "response": "SmartScheduler is our AI calendar assistant."},

    {"instruction": "How much does SmartScheduler cost?",
     "response": "10 euros/month individual, 8 euros/user teams."},

    {"instruction": "How do I reset my password?",
     "response": "Click 'Forgot Password' on the login page."},

    {"instruction": "Can I export my data?",
     "response": "Yes, go to Settings > Data > Export."},
]


# Rephrasings per question (same order as base_examples)
paraphrases = [
    ["What does TechStartup Inc do?", "What is TechStartup Inc's business?",
     "Tell me what TechStartup Inc makes.", "What kind of company is TechStartup Inc?"],
    ["When was TechStartup Inc founded?", "What year was TechStartup Inc started?",
     "How old is TechStartup Inc?", "When did TechStartup Inc begin?"],
    ["Where is TechStartup Inc located?", "Where is TechStartup Inc based?",
     "Which city is TechStartup Inc in?", "Where are TechStartup Inc's offices?"],
    ["What is SmartScheduler?", "Tell me about SmartScheduler.",
     "What does SmartScheduler do?", "Describe SmartScheduler."],
    ["How much does SmartScheduler cost?", "What is SmartScheduler's price?",
     "What does SmartScheduler cost per month?", "How much is a SmartScheduler subscription?"],
    ["How do I reset my password?", "I forgot my password, what do I do?",
     "How can I change my password?", "What is the password reset process?"],
    ["Can I export my data?", "How do I export my data?",
     "Is data export possible?", "Where is the data export option?"],
]

def expand_faq(base_examples, paraphrases, seed=42):
    """Deterministically expand the FAQ pairs, splitting by paraphrase GROUP.

    Each paraphrase plus its four polite-prefix variants forms one group
    (7 questions x 4 paraphrases = 28 groups of 4). We shuffle the GROUPS
    and hold out 5 whole groups (20 examples) BEFORE any train/eval string
    exists in the other split, so no eval phrasing, prefixed or not, was
    ever trained on. Train takes 80 of the remaining 92 examples.
    """
    prefixes = ["", "Please tell me: ", "Quick question: ", "Could you tell me: "]
    groups = []
    for qid, (example, variants) in enumerate(zip(base_examples, paraphrases)):
        for paraphrase in variants:
            groups.append((qid, paraphrase,
                           [{"instruction": p + paraphrase,
                             "response": example["response"]} for p in prefixes]))
    rng = random.Random(seed)
    rng.shuffle(groups)

    # Hold out 5 whole groups, but never the LAST group of a question:
    # every intent must keep at least one phrasing in training.
    left = {qid: sum(1 for q, _, _ in groups if q == qid)
            for qid in range(len(base_examples))}
    eval_groups, train_groups = [], []
    for qid, paraphrase, examples in groups:
        if len(eval_groups) < 5 and left[qid] > 1:
            eval_groups.append((qid, paraphrase, examples))
        else:
            train_groups.append((qid, paraphrase, examples))
        left[qid] -= 1

    eval_data = [ex for _, _, g in eval_groups for ex in g]      # 5 x 4 = 20
    train_pool = [ex for _, _, g in train_groups for ex in g]    # 23 x 4 = 92
    rng.shuffle(train_pool)
    train_data = train_pool[:80]
    return train_data, eval_data

train_data, eval_data = expand_faq(base_examples, paraphrases)

# Assert 1: exact question strings are disjoint (the classic leakage check)
train_questions = {ex["instruction"] for ex in train_data}
assert all(ex["instruction"] not in train_questions for ex in eval_data)

# Assert 2: paraphrase GROUPS are disjoint. Strip the polite prefixes and
# check that no eval phrasing appears in training in ANY prefixed form.
def core_question(instruction):
    for p in ["Please tell me: ", "Quick question: ", "Could you tell me: "]:
        if instruction.startswith(p):
            return instruction[len(p):]
    return instruction

train_cores = {core_question(q) for q in train_questions}
assert all(core_question(ex["instruction"]) not in train_cores for ex in eval_data)

# Here each distinct answer identifies one FAQ intent. Measure coverage in both splits.
train_intents = {ex["response"] for ex in train_data}
eval_intents = {ex["response"] for ex in eval_data}
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")
print(f"Intents covered in training: {len(train_intents)}/7")
print(f"Intents covered in eval: {len(eval_intents)}/7; "
      f"also in training: {len(eval_intents & train_intents)}")
print(f"Eval phrasings seen in training (prefixes stripped): "
      f"{sum(core_question(ex['instruction']) in train_cores for ex in eval_data)}/20")
# Output: Train: 80, Eval: 20
# Output: Intents covered in training: 7/7
# Output: Intents covered in eval: 4/7; also in training: 4
# Output: Eval phrasings seen in training (prefixes stripped): 0/20


def save_as_jsonl(examples, filepath):
    """Save examples in JSONL format (one JSON object per line)."""
    with open(filepath, 'w') as f:
        for example in examples:
            f.write(json.dumps(example) + '\n')

def load_jsonl(filepath):
    """Load examples from JSONL format."""
    examples = []
    with open(filepath, 'r') as f:
        for line in f:
            examples.append(json.loads(line))
    return examples

# Write the 80 training rows first, then the 20 evaluation rows. The packaged
# English reference for this seed is data/faq_dataset.jsonl.
save_as_jsonl(train_data + eval_data, 'faq_dataset.jsonl')


In [ ]:
# ===== FORMATTING + BATCHING (from the book) =====
# Define special tokens
INST_START = "[INST]"
INST_END = "[/INST]"

def build_prompt(instruction):
    """The ONE prompt format shared by training and generation.

    Ends right after [/INST], with no trailing space: a trailing space
    can tokenize differently alone than glued to the answer's first
    word (" Tech"), which would make inference feed the model a prefix
    it never saw in training.
    """
    return f"{INST_START} {instruction} {INST_END}"

def format_example(instruction, response, tokenizer):
    """
    Format an instruction/response pair for training.

    Needs a fast tokenizer (GPT2TokenizerFast, which our setup loads):
    only fast tokenizers report character offsets for each token.

    Returns:
        input_ids: Full tokenized sequence
        labels: Same as input_ids, but with -100 for instruction tokens
    """
    # Format: [INST] instruction [/INST] response + end-of-text.
    # The EOS token teaches the model where answers END, so generation
    # can stop there instead of rambling on after the answer.
    prompt = build_prompt(instruction)
    full_text = prompt + " " + response + tokenizer.eos_token

    # Tokenize, asking for each token's character span in full_text
    enc = tokenizer(full_text, return_offsets_mapping=True)
    input_ids = torch.tensor(enc["input_ids"])

    # Find where the response starts: the first token whose character
    # span lies fully past the prompt. BPE glues the space we added
    # onto the answer's first word (" Tech"), so that token belongs to
    # the answer; counting the prompt's tokens alone would mask it.
    response_start = next(i for i, (_, end) in enumerate(enc["offset_mapping"])
                          if end > len(prompt))

    # Create labels: -100 for instruction tokens (ignored by loss)
    labels = input_ids.clone()
    labels[:response_start] = -100  # Mask instruction tokens

    return input_ids, labels


import torch
from functools import partial
from torch.utils.data import DataLoader

PAD_ID = tokenizer.eos_token_id  # GPT-2 pads with end-of-text

def collate_fn(examples, tokenizer, max_length=128):
    """Tokenize instruction/response pairs into padded input_ids + labels."""
    batch_ids, batch_labels = [], []
    for ex in examples:
        ids, labels = format_example(ex['instruction'], ex['response'], tokenizer)
        ids, labels = ids[:max_length], labels[:max_length]
        pad = max_length - len(ids)
        ids = torch.cat([ids, torch.full((pad,), PAD_ID, dtype=torch.long)])
        labels = torch.cat([labels, torch.full((pad,), -100, dtype=torch.long)])
        batch_ids.append(ids)
        batch_labels.append(labels)
    return {'input_ids': torch.stack(batch_ids),
            'labels': torch.stack(batch_labels)}

collate = partial(collate_fn, tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True, collate_fn=collate)


# Check one batch
batch = next(iter(train_loader))
print(f"Input IDs shape: {batch['input_ids'].shape}")
print(f"Labels shape: {batch['labels'].shape}")
print(f"Masked positions in first example: {(batch['labels'][0] == -100).sum().item()}")

## 4. Baseline Evaluation (BEFORE Fine-Tuning)

Let's see how the base model responds to our FAQ prompts.

In [ ]:
@torch.no_grad()
def generate_response(model, prompt, tokenizer, max_new_tokens=50, temperature=0.8):
    """Generate a response to an instruction prompt."""
    model.eval()
    full_prompt = build_prompt(prompt)
    input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)
    
    output_ids = model.generate(
        input_ids, 
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True
    )
    
    response = tokenizer.decode(output_ids[0][len(input_ids[0]):])
    return response.strip()

# Test on FAQ prompts
test_prompts = [
    "What does TechStartup Inc do?",
    "How do I reset my password?",
    "What is SmartScheduler?"
]

print(f"BEFORE FINE-TUNING ({BASE_SOURCE}):")
print("="*60)
for prompt in test_prompts:
    response = generate_response(base_model, prompt, tokenizer)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:100]}...")
print("\n(Shakespeare-flavored gibberish - the Chapter 12 base knows")
print("nothing about TechStartup Inc, and a 16M from-scratch model")
print("never learned fluent English. Honest expectations!)")

## 5. Define LoRA Components

Build the LoRA adapter layer that adds trainable parameters without modifying base weights.

### Why LoRA Instead of Full Fine-Tuning?

Full fine-tuning has a problem: **catastrophic forgetting**. When you update all weights for a new task, the model can "forget" what it learned during pre-training. LoRA greatly reduces this by keeping base weights *frozen* and adding small trainable matrices *alongside* them.

Think of it like reading glasses: your eyes (base model) stay the same, but the glasses (LoRA) add a small correction for specific tasks.

In [ ]:
# ===== LORA LINEAR LAYER (paper convention: A down, B up) =====
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """
    A linear layer with Low-Rank Adaptation.

    Think of this like adding a small "correction note" to an existing layer.
    The base layer stays frozen, while two small matrices learn adjustments.
    """

    def __init__(self, in_features, out_features, rank=8, alpha=16):
        super().__init__()

        # The original layer (we'll freeze this); bias matches GPT-2's qkv_proj
        self.base = nn.Linear(in_features, out_features)

        # LoRA matrices (paper convention: A down, B up)
        # A: (in_features, rank) - "down" projection
        # B: (rank, out_features) - "up" projection
        self.lora_A = nn.Parameter(torch.zeros(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))

        # Initialize A with small random values (Kaiming), B with zeros.
        # Init through the transposed view so Kaiming sees (rank, in_features),
        # the weight shape of nn.Linear(in_features, rank): fan_in becomes
        # in_features. Writing in place on a view updates lora_A itself.
        nn.init.kaiming_uniform_(self.lora_A.T, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        # Scaling factor
        self.scale = alpha / rank

    def freeze_base(self):
        """Freeze the base layer so only LoRA matrices are trained."""
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

    def forward(self, x):
        # x shape: (batch, sequence, in_features)

        # Original output from frozen weights
        base_output = self.base(x)  # (batch, seq, out_features)

        # LoRA path: x @ A @ B, scaled
        # x @ A: (batch, seq, in_features) @ (in_features, rank)
        #      = (batch, seq, rank)  [compressed!]
        # ... @ B: (batch, seq, rank) @ (rank, out_features)
        #        = (batch, seq, out_features)  [expanded back]
        lora_output = (x @ self.lora_A @ self.lora_B) * self.scale

        return base_output + lora_output

print("LoRALinear defined!")

# Demonstrate parameter savings
in_features, out_features, rank = 256, 256, 8
full_params = in_features * out_features
lora_params = in_features * rank + rank * out_features

print(f"\nParameter comparison (256x256 layer):")
print(f"  Full fine-tuning: {full_params:,} parameters")
print(f"  LoRA (rank=8):    {lora_params:,} parameters")
print(f"  Savings:          {full_params/lora_params:.1f}x fewer!")

## 6. Apply LoRA to Model

Replace attention projections with LoRA versions and freeze the base model.

In [ ]:
def add_lora_to_model(model, rank=8, alpha=16):
    """
    Add LoRA to attention projections.

    We target qkv_proj, the combined query/key/value projection.
    The adapter learns joint corrections for Q, K, and V.
    """
    # Step 1: Freeze ALL parameters first
    for param in model.parameters():
        param.requires_grad = False

    # Step 2: Replace qkv_proj in each transformer block
    # We iterate explicitly through the blocks we built in Chapter 11
    for block in model.blocks:
        # Get the attention layer's QKV projection
        original_qkv = block.attn.qkv_proj
        in_features = original_qkv.in_features
        out_features = original_qkv.out_features

        # Create LoRA-enhanced replacement on the same device as the model
        lora_qkv = LoRALinear(in_features, out_features, rank, alpha)
        lora_qkv = lora_qkv.to(original_qkv.weight.device)

        # Copy pretrained weights (and bias!) to the base layer
        lora_qkv.base.weight.data = original_qkv.weight.data.clone()
        lora_qkv.base.bias.data = original_qkv.bias.data.clone()

        # Freeze the base (LoRA matrices stay trainable)
        lora_qkv.freeze_base()
        if lora_qkv.base.bias is not None:
            lora_qkv.base.bias.requires_grad = False

        # Replace the layer
        block.attn.qkv_proj = lora_qkv

    return model


# Same base as the book: the Chapter 12 checkpoint imported above.
# Baseline, training, and reload all share these exact weights, so the
# BEFORE/AFTER comparisons stay honest. (If Route B mini-trained the base,
# losses will differ from the book's 3-epoch-checkpoint numbers.)
set_seed(42)  # BEFORE adapter init and batch shuffling (seeding after training reproduces nothing)
model = MiniGPT(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model = add_lora_to_model(model, rank=8, alpha=16)

# Count parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Total parameters:     {total:,}")
print(f"Trainable (LoRA):     {trainable:,}")
print(f"Trainable percentage: {100*trainable/total:.2f}%")

## 7. Fine-Tuning Loop

Train using the 5-step recipe with masked loss.

In [ ]:
import torch.nn.functional as F

def compute_masked_loss(logits, labels):
    """
    Compute cross-entropy loss, ignoring positions where labels == -100.

    This is like grading only the answer portion of an exam,
    not the question that was copied from the prompt.
    """
    # Shift for next-token prediction (same as Chapter 12)
    # .contiguous() ensures memory is laid out sequentially for .view()
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    # Cross-entropy with ignore_index=-100
    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100  # PyTorch ignores these positions
    )

    return loss


def finetune_epoch(model, dataloader, optimizer, device):
    """
    Fine-tune for one epoch using the 5-step recipe with masked loss.
    """
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        # ===== THE 5-STEP RECIPE =====

        # Step 1: Zero gradients
        optimizer.zero_grad()

        # Step 2: Forward pass
        logits = model(input_ids)

        # Step 3: Compute MASKED loss
        loss = compute_masked_loss(logits, labels)

        # Step 4: Backward pass
        loss.backward()

        # Step 5: Update weights
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

print("Training functions defined!")

In [ ]:
# ===== TRAINING =====
num_epochs = 3
learning_rate = 1e-3  # LoRA adapters tolerate a higher LR than full fine-tuning

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate  # same optimizer as the book: plain AdamW, no weight decay
)

train_losses = []
print(f"Training on {len(train_data)} examples for {num_epochs} epochs\n")
for epoch in range(num_epochs):
    loss = finetune_epoch(model, train_loader, optimizer, device)
    train_losses.append(loss)
    print(f"Epoch {epoch+1}: masked loss {loss:.4f}")

print("\nFine-tuning complete!")
print("Note: with our from-scratch 16M Shakespeare base, expect gibberish generations;")
print("the win is the loss drop and the adapter mechanics. For fluent answers,")
print("start from pretrained GPT-2 weights (Chapter 11's loader).")

## 8. Plot Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Training Loss', fontsize=12)
plt.title('Fine-Tuning Loss Curve', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(range(1, len(train_losses) + 1))
plt.show()

print(f"\nLoss decreased from {train_losses[0]:.4f} to {train_losses[-1]:.4f}")
print(f"Improvement: {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

## 9. Evaluation (AFTER Fine-Tuning)

The honest part: measuring what changed. With our from-scratch 16M base model, generations stay gibberish before AND after (it never learned fluent English). The measurable win is the training-loss drop and the adapter mechanics. A pretrained base like GPT-2 (Chapter 11's loader) is what produces a fluent before/after.

In [ ]:
def evaluate_model(model, test_examples, tokenizer, device):
    """
    Evaluate fine-tuned model on test examples.
    """
    model.eval()
    results = []

    for example in test_examples:
        instruction = example['instruction']
        expected = example['response']

        # Generate from the SAME prefix format training used
        prompt = build_prompt(instruction)
        input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

        # Held-out loss: teacher-force the CORRECT answer through the
        # model and grade only response tokens (same masked loss as
        # training, but on data the adapters never saw)
        full_ids, full_labels = format_example(instruction, expected, tokenizer)
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(device))
            resp_loss = compute_masked_loss(logits, full_labels.unsqueeze(0).to(device))

        output = model.generate(
            input_ids,
            max_new_tokens=50,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )

        # Decode only the new tokens, dropping the EOS terminator itself
        new_tokens = output[0][input_ids.shape[1]:].tolist()
        if tokenizer.eos_token_id in new_tokens:
            new_tokens = new_tokens[:new_tokens.index(tokenizer.eos_token_id)]
        generated = tokenizer.decode(new_tokens).strip()

        # Exact match
        is_exact = generated.lower() == expected.lower()

        # Word overlap
        gen_words = set(generated.lower().split())
        exp_words = set(expected.lower().split())
        overlap = len(gen_words & exp_words) / max(len(exp_words), 1)

        results.append({
            'instruction': instruction,
            'expected': expected,
            'generated': generated,
            'exact_match': is_exact,
            'word_overlap': overlap,
            'response_loss': resp_loss.item()
        })

    # Summary
    accuracy = sum(r['exact_match'] for r in results) / len(results)
    avg_overlap = sum(r['word_overlap'] for r in results) / len(results)

    return {
        'exact_match_accuracy': accuracy,
        'average_word_overlap': avg_overlap,
        'avg_response_loss': sum(r['response_loss'] for r in results) / len(results),
        'detailed_results': results
    }

# Evaluate on the held-out split
print("Evaluating on test set...")
results = evaluate_model(model, eval_data, tokenizer, device)

print(f"\n===== EVALUATION RESULTS =====")
print(f"Exact Match Accuracy: {results['exact_match_accuracy']*100:.1f}%")
print(f"Average Word Overlap: {results['average_word_overlap']*100:.1f}%")
print("(Near 0% is expected with a from-scratch base model; see the note above.)")
print(f"Held-out response loss: {results['avg_response_loss']:.2f}")


In [ ]:
# Show detailed results
print("\n===== DETAILED TEST RESULTS =====")
for r in results['detailed_results']:
    status = "✓" if r['exact_match'] else "○"
    print(f"\n{status} Q: {r['instruction']}")
    print(f"  Expected:  {r['expected']}")
    print(f"  Generated: {r['generated'][:100]}..." if len(r['generated']) > 100 else f"  Generated: {r['generated']}")
    print(f"  Overlap:   {r['word_overlap']*100:.0f}%")

In [ ]:
def show_before_after(base_model, finetuned_model, test_prompts, tokenizer, device):
    """
    Side-by-side comparison: base model vs fine-tuned model.
    """
    print("=" * 70)
    print("BEFORE vs AFTER Fine-Tuning")
    print("=" * 70)

    for prompt in test_prompts:
        full_prompt = build_prompt(prompt)
        input_ids = tokenizer.encode(full_prompt, return_tensors='pt').to(device)

        base_output = base_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)
        ft_output = finetuned_model.generate(
            input_ids, max_new_tokens=50,
            eos_token_id=tokenizer.eos_token_id)

        base_text = tokenizer.decode(base_output[0][len(input_ids[0]):])
        ft_text = tokenizer.decode(ft_output[0][len(input_ids[0]):])

        print(f"\nPrompt: {prompt}")
        print(f"  BEFORE: {base_text.strip()}")
        print(f"  AFTER:  {ft_text.strip()}")
        print("-" * 70)

# BEFORE model: an exact copy of the baseline weights (unadapted)
base_model_fresh = MiniGPT(config).to(device)
base_model_fresh.load_state_dict(base_model.state_dict())

test_prompts = ["What does TechStartup Inc do?", "How do I reset my password?"]
set_seed(42)  # reseed so the sampled comparison below is reproducible
show_before_after(base_model_fresh, model, test_prompts, tokenizer, device)

print("\nBoth sides are gibberish: a 16M from-scratch model never learned fluent")
print("English. The measurable win is the loss drop + working adapter mechanics.")
print("Load GPT-2 weights first (Chapter 11) to see a fluent before/after.")

## 10. Save and Load LoRA Weights

In [ ]:
def save_lora_weights(model, filepath):
    """
    Save only the LoRA parameters (tiny file!).
    """
    lora_state_dict = {
        name: param for name, param in model.state_dict().items()
        if 'lora_' in name
    }
    torch.save(lora_state_dict, filepath)

    # Show file size
    import os
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"LoRA weights saved: {filepath} ({size_mb:.2f} MB)")



# Save LoRA weights
save_lora_weights(model, 'techstartup_lora.pt')

# Compare to full model size
torch.save(model.state_dict(), 'full_model.pt')
full_size = os.path.getsize('full_model.pt') / (1024 * 1024)
lora_size = os.path.getsize('techstartup_lora.pt') / (1024 * 1024)
print(f"Full model saved: full_model.pt ({full_size:.1f} MB)")
print(f"\nLoRA is {full_size/lora_size:.0f}x smaller than the full model!")

In [ ]:
def load_lora_weights(model, filepath):
    """
    Load LoRA weights into a model that already has LoRA layers.
    """
    lora_state_dict = torch.load(filepath, weights_only=True)
    model.load_state_dict(lora_state_dict, strict=False)
    print(f"LoRA weights loaded from: {filepath}")



# Demonstrate loading
print("\n===== Testing Load/Save =====")
# The adapter is only meaningful on the exact base it was trained on:
# reload best_model.pt FIRST, then add LoRA structure, then the adapter.
new_model = MiniGPT(config).to(device)
new_model.load_state_dict(checkpoint["model_state_dict"])  # base first!
new_model = add_lora_to_model(new_model, rank=8, alpha=16)

before_load = generate_response(new_model, "What does TechStartup Inc do?", tokenizer)
print(f"Before loading LoRA: {before_load[:60]}...")

load_lora_weights(new_model, 'techstartup_lora.pt')

after_load = generate_response(new_model, "What does TechStartup Inc do?", tokenizer)
print(f"After loading LoRA:  {after_load[:60]}...")
# Sanity check: reloaded model must produce the same logits as the trained one
probe = tokenizer.encode("[INST] What is SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval(); new_model.eval()
with torch.no_grad():
    torch.testing.assert_close(new_model(probe), model(probe), atol=1e-5, rtol=0)
print("Reload parity check passed")
print("\nLoRA adapters successfully saved and loaded!")


# ===== MERGE: fold LoRA into the base and REMOVE the wrappers =====
def merge_lora_into_base(model):
    """
    Merge LoRA into base weights and REMOVE the adapter wrappers,
    so the merged model is a plain MiniGPT again (permanent).
    """
    for block in model.blocks:
        lora = block.attn.qkv_proj
        if isinstance(lora, LoRALinear):
            # Fold the correction into the base weight: W + (A @ B * scale).T
            delta = lora.lora_A @ lora.lora_B * lora.scale
            lora.base.weight.data += delta.T  # Transpose for Linear convention
            # Swap the wrapper out for the plain merged layer
            block.attn.qkv_proj = lora.base

    print("LoRA weights merged into base model")

# Merge and verify: same logits, adapter parameters gone.
# NOTE: merging is permanent; run this cell only after the comparisons above.
probe = tokenizer.encode("[INST] What is SmartScheduler? [/INST]",
                         return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    adapter_logits = model(probe)

merge_lora_into_base(model)

with torch.no_grad():
    merged_logits = model(probe)
torch.testing.assert_close(merged_logits, adapter_logits, atol=1e-5, rtol=0)

total = sum(p.numel() for p in model.parameters())
print(f"Merged model parameters: {total:,} (adapters removed)")
assert not any(isinstance(m, LoRALinear) for m in model.modules())
print("Merge parity check passed")
# Why the transpose? forward computes x @ A @ B, so the update is (in, out);
# nn.Linear stores weights as (out, in), hence delta.T. Replacing the wrapper
# (not just zeroing the adapters) removes the two dead adapter matmuls.


## Summary

**What we built:**

1. **Instruction/response dataset** with loss masking
2. **LoRA adapters** that train <1% of parameters
3. **Fine-tuning loop** using the 5-step recipe
4. **Evaluation** comparing before/after behavior (honestly: loss drops, text stays gibberish on a from-scratch base)
5. **Adapter saving/loading** for deployment

**Key insights:**

- Loss masking focuses training on what matters (responses, not instructions)
- The parameter saving depends on the layer dimensions and rank; count the trainable weights in your own model.
- Freezing base weights greatly reduces **catastrophic forgetting** (losing pre-trained knowledge)
- Quality of data matters more than quantity for fine-tuning
- Adapters are tiny and can be swapped for different tasks

**Next:** Chapter 14 will explore prompt engineering - getting better outputs without changing model weights at all!

## Optional Experiment: Random-Base Variant (clearly labelled)

The main flow above uses the Chapter 12 checkpoint everywhere. This experiment repeats the LoRA recipe on a **randomly initialized base** to show why the checkpoint matters. It is NOT the book's run: baseline, training, and reload here all share the same random weights, kept separate from the checkpoint flow. Expect a different (usually higher) starting loss and different generations.

In [ ]:
# ===== RANDOM-BASE VARIANT (labelled experiment, NOT the book's run) =====
set_seed(42)
rand_base = MiniGPT(config).to(device)  # random init, NO checkpoint
rand_model = MiniGPT(config).to(device)
rand_model.load_state_dict(rand_base.state_dict())  # same random weights
rand_model = add_lora_to_model(rand_model, rank=8, alpha=16)

rand_optimizer = AdamW([p for p in rand_model.parameters() if p.requires_grad],
                       lr=1e-3, weight_decay=0.01)
rand_losses = []
for epoch in range(3):
    loss = finetune_epoch(rand_model, train_loader, rand_optimizer, device)
    rand_losses.append(loss)
    print(f"[random base] Epoch {epoch+1}: masked loss {loss:.4f}")

print(f"\nCheckpoint-base losses: {[f'{l:.4f}' for l in train_losses]}")
print(f"Random-base losses:     {[f'{l:.4f}' for l in rand_losses]}")
print("Same recipe, different base: the numbers depend on the starting weights,")
print("which is exactly why the adapter must be reloaded onto its OWN base.")

## Exercises

### Exercise 1: Different LoRA Ranks

Try rank=4 and rank=16. How does it affect training and results?

In [ ]:
# YOUR CODE HERE
# 1. Create model with rank=4
# 2. Train for 3 epochs
# 3. Compare loss and evaluation to rank=8

### Exercise 2: Create Your Own FAQ

Create a dataset about a topic you know (your school, hobby, etc).

In [ ]:
# YOUR CODE HERE
# 1. Create 30+ instruction/response pairs
# 2. Fine-tune the model
# 3. Test with your own questions

### Exercise 3: Loss Masking Ablation

What happens if we DON'T mask the instruction tokens?

In [ ]:
# YOUR CODE HERE
# 1. Modify the dataset to NOT mask instruction tokens (all labels = actual tokens)
# 2. Train the model
# 3. Compare results to masked version
# 4. What do you observe?